# How many tweets did the CN actually help vs. hurt, one by one?

**Data used:** the experiment itself (who got a CN reply + Views/Likes/Shares)

**Short answer:** 62% of CN’d tweets gained views. The result also holds no matter which way we normalise it.



# Saveski-Inspired Analysis

**Inspired by:** Saveski et al. 2025 — *Community Notes suppress engagement on X* (PNAS)

## Analyses implemented

| # | Analysis | What it adds |
|---|----------|--------------|
| 1 | **Individual effect distribution** | % of Treatment tweets with CATE > 0. Saveski reports 43% showed increased views. |
| 2 | **Growth vs. Overall normalization** | Does anti-suppression hold under both framings? |
| 3 | **Detection lag quartile table** | Tercile analysis reframed as 4 quartiles (Saveski format). |

**Sign convention:** r < 0 = Treatment grew more = **anti-suppression**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then take the field-experiment folder inside it.
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
BASE_DIR = _root / "field-experiment"
if not (BASE_DIR / "data").is_dir():
    raise RuntimeError(
        "Could not locate the field-experiment folder from " + str(Path.cwd()) +
        ". Run this notebook from inside the cloned repository."
    )
DATA_DIR   = BASE_DIR / "data"
FEAT_DIR   = BASE_DIR / "data"
OUT_DIR    = BASE_DIR / "outputs" / "individual_effects"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_CSV  = FEAT_DIR / "tweet_features.csv"
MONITORING_XL = DATA_DIR / "Tweet Monitoring.xlsx"
CONTROL_XL    = DATA_DIR / "Control_Group.xlsx"
TREATMENT_XL  = DATA_DIR / "Treatment_Group.xlsx"

METRICS     = ["Views", "Likes", "Shares"]
MAIN_WINDOW = 13
N_BOOTSTRAP = 5000
RANDOM_SEED = 42

print("Config loaded.")

In [ ]:
def rank_biserial_r(trt, ctrl):
    U, _ = stats.mannwhitneyu(trt, ctrl, alternative="two-sided")
    return 1 - (2 * U) / (len(trt) * len(ctrl))

def bootstrap_ci(trt, ctrl, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    boots = [
        rank_biserial_r(
            rng.choice(trt,  size=len(trt),  replace=True),
            rng.choice(ctrl, size=len(ctrl), replace=True),
        )
        for _ in range(n_boot)
    ]
    return np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def compute_effect(trt, ctrl, min_n=20):
    trt  = np.asarray(trt.dropna()  if hasattr(trt,  "dropna") else trt)
    ctrl = np.asarray(ctrl.dropna() if hasattr(ctrl, "dropna") else ctrl)
    if len(trt) < min_n or len(ctrl) < min_n:
        return None
    _, p = stats.mannwhitneyu(trt, ctrl, alternative="two-sided")
    r    = rank_biserial_r(trt, ctrl)
    lo, hi = bootstrap_ci(trt, ctrl)
    sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else ("." if p<0.10 else "")))
    return {"r":r,"ci_lo":lo,"ci_hi":hi,"p":p,"n_trt":len(trt),"n_ctrl":len(ctrl),"sig":sig}

print("Helpers defined.")

---
## 1. Data Loading

Same monitoring pipeline. Also loads Control/Treatment group files
for `Detection_Date` and `Creation_Date` to compute `Detection_Lag_Minutes`.

In [ ]:
# ── Parse monitoring dates ────────────────────────────────────────────────────
def parse_mixed_date(date_val):
    if pd.isna(date_val): return pd.NaT
    date_str = str(date_val).strip()
    def is_valid(year, month):
        return (year == 2025 and month == 12) or (year == 2026 and month == 1)
    if "-" in date_str and date_str[:4].isdigit():
        parts = date_str.split("-")
        year = int(parts[0]); num1 = int(parts[1]); num2 = int(parts[2].split()[0])
        if is_valid(year, num1): month, day = num1, num2
        elif is_valid(year, num2): month, day = num2, num1
        else: month, day = num1, num2
        return pd.Timestamp(year=year, month=month, day=day)
    elif "/" in date_str:
        parts = date_str.split("/")
        num1 = int(parts[0]); num2 = int(parts[1]); year = int(parts[2].split()[0])
        if is_valid(year, num2): day, month = num1, num2
        elif is_valid(year, num1): month, day = num1, num2
        else: day, month = num1, num2
        return pd.Timestamp(year=year, month=month, day=day)
    else:
        return pd.to_datetime(date_val, errors="coerce")

# ── Load features + monitoring ────────────────────────────────────────────────
feat = pd.read_csv(FEATURES_CSV)
print(f"tweet_features.csv: {feat.shape}")

monitoring = pd.read_excel(MONITORING_XL)
monitoring["Start_Date (Creation)"] = monitoring["Start_Date (Creation)"].apply(parse_mixed_date)
monitoring["Sample_Date"]           = monitoring["Sample_Date"].apply(parse_mixed_date)
monitoring["Day"] = (monitoring["Sample_Date"] - monitoring["Start_Date (Creation)"]).dt.days

pivoted = monitoring.pivot_table(
    index="URL", columns="Day",
    values=["Views", "Likes", "Comments", "Shares"], aggfunc="first"
)
pivoted.columns = [f"{m}_Day{d}" for m, d in pivoted.columns]
pivoted = pivoted.reset_index()
print(f"Pivoted monitoring: {pivoted.shape}")

df = feat.merge(pivoted, on="URL", how="left")
print(f"After merge: {df.shape}")

for metric in ["Likes", "Shares", "Comments"]:
    col = f"{metric}_Day0"
    if col in df.columns:
        df[col] = df[col].fillna(0)

for metric in METRICS:
    d0 = f"{metric}_Day0"
    dN = f"{metric}_Day{MAIN_WINDOW}"
    if d0 in df.columns and dN in df.columns:
        df[f"{metric}_Growth_{MAIN_WINDOW}d"] = (df[dN] - df[d0]) / (df[d0] + 1) * 100
        df[f"log_baseline_{metric}"] = np.log1p(df[d0])

df["is_treatment"] = (df["Group"] == "Treatment").astype(int)
print("Growth DVs computed:")
for metric in METRICS:
    col = f"{metric}_Growth_{MAIN_WINDOW}d"
    print(f"  {col:<30} {df[col].notna().sum()} / {len(df)}")

In [ ]:
# ── Detection lag from Control/Treatment group files ─────────────────────────
ctrl_xl = pd.read_excel(CONTROL_XL,   usecols=["URL", "Detection_Date", "Creation_Date"])
trt_xl  = pd.read_excel(TREATMENT_XL, usecols=["URL", "Detection_Date", "Creation_Date"])
dates   = pd.concat([ctrl_xl, trt_xl], ignore_index=True).drop_duplicates("URL")

dates["Detection_Date"] = pd.to_datetime(dates["Detection_Date"], dayfirst=True, errors="coerce")
dates["Creation_Date"]  = pd.to_datetime(dates["Creation_Date"],  dayfirst=True, errors="coerce")
dates["Detection_Lag_Minutes"] = (
    (dates["Detection_Date"] - dates["Creation_Date"]).dt.total_seconds() / 60
)

df = df.merge(dates[["URL", "Detection_Lag_Minutes"]], on="URL", how="left")

lag = df["Detection_Lag_Minutes"].dropna()
print(f"Detection_Lag_Minutes: n={len(lag)}  "
      f"min={lag.min():.0f}  median={lag.median():.0f}  "
      f"mean={lag.mean():.0f}  max={lag.max():.0f} min")
print(f"Missing lag: {df['Detection_Lag_Minutes'].isna().sum()}")

ctrl = df[df["Group"] == "Control"]
trt  = df[df["Group"] == "Treatment"]

---
## 2. Analysis 1 — Individual Effect Distribution

**Saveski 2025:** Per-post synthetic control estimates show **43% of fact-checked posts had increased views** after note attachment — i.e., even a suppression intervention left many posts unaffected.

**Our approach — within-stratum matched counterfactual:**
1. Bin all tweets by `Views_Day0` into quintiles
2. Each Treatment tweet's estimated CATE = its growth − median Control growth in the same quintile
3. Report % CATE > 0 (anti-suppressed) per metric

This is descriptive. The mean of these CATEs should match the direction of the MWU r from the main analysis.

In [ ]:
N_QUINTILES = 5

df["views_quintile"] = pd.qcut(
    df["Views_Day0"].rank(method="first"),
    q=N_QUINTILES, labels=[f"Q{i+1}" for i in range(N_QUINTILES)]
)

print("Views_Day0 quintile boundaries:")
print(df.groupby("views_quintile", observed=True)["Views_Day0"]
        .agg(["min","median","max","count"]))
print()

cate_results = {}

print("=" * 60)
print("PER-TWEET CATE ESTIMATES (within-quintile counterfactual)")
print("=" * 60)

for metric in METRICS:
    col = f"{metric}_Growth_{MAIN_WINDOW}d"
    ctrl_medians = (
        df[df["Group"] == "Control"]
        .groupby("views_quintile", observed=True)[col]
        .median()
    )
    trt_df = df[df["Group"] == "Treatment"].copy()
    trt_df["ctrl_counterfactual"] = trt_df["views_quintile"].map(ctrl_medians).astype(float)
    trt_df["cate"] = trt_df[col] - trt_df["ctrl_counterfactual"]
    trt_df = trt_df.dropna(subset=["cate"])

    pct_pos  = (trt_df["cate"] > 0).mean() * 100
    pct_neg  = (trt_df["cate"] < 0).mean() * 100
    pct_zero = (trt_df["cate"] == 0).mean() * 100
    cate_results[metric] = trt_df["cate"].values

    print(f"  {metric} (n={len(trt_df)})")
    print(f"    CATE > 0 (anti-suppressed): {pct_pos:.1f}%")
    print(f"    CATE = 0 (no change):       {pct_zero:.1f}%")
    print(f"    CATE < 0 (suppressed):      {pct_neg:.1f}%")
    print(f"    Mean CATE:   {trt_df['cate'].mean():.2f} pp")
    print(f"    Median CATE: {trt_df['cate'].median():.2f} pp")

print()
print("Saveski 2025: 43% of posts showed INCREASED views (despite overall suppression).")
print("Our anti-suppression finding predicts >50% CATE > 0 for Views.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, METRICS):
    cates = cate_results[metric]
    p1, p99 = np.percentile(cates, 1), np.percentile(cates, 99)
    cates_c = np.clip(cates, p1, p99)
    pct_pos = (cates > 0).mean() * 100

    ax.hist(cates_c, bins=40, color="#4C72B0", alpha=0.75, edgecolor="white")
    ax.axvline(0, color="black", linewidth=1.5, linestyle="--")
    ax.axvline(np.median(cates), color="#C44E52", linewidth=2,
               label=f"Median={np.median(cates):.1f} pp")

    ymax = ax.get_ylim()[1]
    if ymax == 0: ymax = 10
    ax.fill_betweenx([0, ymax], 0, max(cates_c.max(), 0.01),
                     alpha=0.08, color="green", label=f"{pct_pos:.0f}% anti-suppressed")
    ax.fill_betweenx([0, ymax], min(cates_c.min(), -0.01), 0,
                     alpha=0.08, color="red",   label=f"{100-pct_pos:.0f}% suppressed")

    ax.set_xlabel("Est. CATE (pp above control stratum)", fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Individual Effect Distribution (within-quintile matched counterfactual)\n"
    "Green = CATE > 0  |  Red = CATE < 0",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(OUT_DIR / "individual_cate_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

---
## 3. Analysis 2 — Growth vs. Overall Normalization

**Saveski 2025** distinguishes two framings of the same outcome:
- **Growth:** additional engagement gained post-intervention (our existing metric)
- **Overall:** final level relative to baseline — a multiplier

Our Growth = `(Day13 − Day0) / (Day0 + 1) × 100`
Overall = `Day13 / (Day0 + 1)`

If both agree in direction and significance → effect is robust to normalization choice.

In [ ]:
for metric in METRICS:
    d0 = f"{metric}_Day0"
    dN = f"{metric}_Day{MAIN_WINDOW}"
    if d0 in df.columns and dN in df.columns:
        df[f"{metric}_Overall"] = df[dN] / (df[d0] + 1)

ctrl = df[df["Group"] == "Control"]
trt  = df[df["Group"] == "Treatment"]

print("=" * 75)
print("GROWTH vs. OVERALL NORMALIZATION")
print("=" * 75)
print(f"  {'Metric':<10} {'Norm':<10} {'Ctrl med':>10} {'Trt med':>10} {'r':>8} {'p':>8} {'sig':>5}")
print("-" * 65)

comparison_rows = []
for metric in METRICS:
    for norm_label, col in [
        ("Growth",  f"{metric}_Growth_{MAIN_WINDOW}d"),
        ("Overall", f"{metric}_Overall"),
    ]:
        c_vals = ctrl[col].dropna().values
        t_vals = trt[col].dropna().values
        p99 = np.percentile(np.concatenate([c_vals, t_vals]), 99)
        c_vals = np.clip(c_vals, None, p99)
        t_vals = np.clip(t_vals, None, p99)
        eff = compute_effect(t_vals, c_vals)
        if eff:
            print(f"  {metric:<10} {norm_label:<10} "
                  f"{np.median(c_vals):>10.3f} {np.median(t_vals):>10.3f} "
                  f"{eff['r']:>+8.3f} {eff['p']:>8.4f} {eff['sig']:>5}")
            comparison_rows.append({"metric":metric,"norm":norm_label,**eff,
                                    "ctrl_med":np.median(c_vals),"trt_med":np.median(t_vals)})

print()
print("If Growth and Overall agree in direction -> robust to normalization choice.")

---
## 4. Analysis 3 — Detection Lag Quartile Table

**Saveski 2025:** Community Notes attached faster produce **stronger suppression**.

`Detection_Lag_Minutes` = time from tweet creation to our detection (CN posted shortly after).
Main notebook Section 4.4 used **terciles**. Here we use **4 quartiles** to match Saveski.

**Q1 = fastest CN deployment (smallest lag), Q4 = slowest.**

In [ ]:
N_BINS = 8
BIN_LABELS = [f"Q{i+1}" for i in range(N_BINS)]
QUANTILE_CUTS = [i / N_BINS for i in range(1, N_BINS)]

df_lag = df.dropna(subset=["Detection_Lag_Minutes"]).copy()

q_bounds = df_lag["Detection_Lag_Minutes"].quantile(QUANTILE_CUTS)
print(f"Detection lag bin boundaries (minutes), {N_BINS} bins:")
for i, (q, v) in enumerate(q_bounds.items()):
    print(f"  Q{i+1}/Q{i+2} boundary: {v:.0f} min")

df_lag["Lag_Bin"] = pd.qcut(
    df_lag["Detection_Lag_Minutes"],
    q=N_BINS, labels=BIN_LABELS
)

print("\nTweets per bin x group:")
print(df_lag.groupby(["Lag_Bin", "Group"], observed=True).size().unstack(fill_value=0))

lag_results = []

print("\n" + "=" * 90)
print(f"DETECTION LAG ({N_BINS} BINS)  (r < 0 = anti-suppression;  Q1=fastest, Q{N_BINS}=slowest)")
print("=" * 90)

for metric in METRICS:
    col = f"{metric}_Growth_{MAIN_WINDOW}d"
    print(f"\n  {metric}")
    print(f"  {'Bin':<5} {'Lag range (min)':>17} {'N_ctrl':>8} {'N_trt':>8} "
          f"{'r':>8} {'CI_lo':>8} {'CI_hi':>8} {'p':>8} {'sig':>5}")
    print(f"  {'-'*82}")
    for q in BIN_LABELS:
        sub      = df_lag[df_lag["Lag_Bin"] == q]
        ctrl_sub = sub[sub["Group"] == "Control"]
        trt_sub  = sub[sub["Group"] == "Treatment"]
        lag_label = (f"{sub['Detection_Lag_Minutes'].min():.0f}"
                     f"-{sub['Detection_Lag_Minutes'].max():.0f}")
        eff = compute_effect(trt_sub[col].dropna(), ctrl_sub[col].dropna())
        if eff:
            print(f"  {q:<5} {lag_label:>17} {eff['n_ctrl']:>8} {eff['n_trt']:>8} "
                  f"{eff['r']:>+8.3f} {eff['ci_lo']:>8.3f} {eff['ci_hi']:>8.3f} "
                  f"{eff['p']:>8.4f} {eff['sig']:>5}")
            lag_results.append({
                "metric": metric, "bin": q,
                "lag_min": sub["Detection_Lag_Minutes"].min(),
                "lag_max": sub["Detection_Lag_Minutes"].max(),
                **eff
            })
        else:
            print(f"  {q:<5} {lag_label:>17}   SKIP (n too small)")

lag_df = pd.DataFrame(lag_results)


In [ ]:
if len(lag_df) == 0:
    print("No results to plot.")
else:
    # Gradient from blue (fast) to red (slow)
    import matplotlib.cm as cm
    cmap = cm.get_cmap("RdYlBu_r", N_BINS)
    bin_colors = {f"Q{i+1}": cmap(i) for i in range(N_BINS)}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

    for ax, metric in zip(axes, METRICS):
        rows = lag_df[lag_df["metric"] == metric].sort_values("bin")
        y_pos = list(range(len(rows)))

        for y, (_, row) in zip(y_pos, rows.iterrows()):
            color = bin_colors.get(row["bin"], "gray")
            ax.plot([row["ci_lo"], row["ci_hi"]], [y, y], color=color, linewidth=2)
            marker = "D" if row["p"] < 0.05 else "o"
            ax.plot(row["r"], y, marker=marker, color=color, markersize=9, zorder=5)
            ax.text(row["ci_hi"] + 0.01, y, f" {row['sig']}", va="center", fontsize=8)

        ax.axvline(0, color="black", linewidth=1, linestyle="--")
        ax.set_yticks(y_pos)
        ax.set_yticklabels(
            [f"{r['bin']} ({r['lag_min']:.0f}-{r['lag_max']:.0f} min)"
             for _, r in rows.iterrows()],
            fontsize=7
        )
        ax.set_xlabel("Rank-biserial r\n(<- anti-suppression | suppression ->)", fontsize=9)
        ax.set_title(metric, fontsize=12, fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
        ax.set_xlim(-0.5, 0.5)

    from matplotlib.lines import Line2D
    axes[-1].legend(handles=[
        Line2D([0], [0], marker="D", color="gray", markersize=8,
               linestyle="None", label="p<0.05"),
        Line2D([0], [0], marker="o", color="gray", markersize=8,
               linestyle="None", label="p>=0.05"),
    ], loc="lower right", fontsize=8)

    plt.suptitle(
        f"CN Effect by Detection Lag ({N_BINS} bins, Q1=fastest, Q{N_BINS}=slowest)",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / "lag_bin_forest.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved.")


---
## 5. Summary Comparison vs. Saveski 2025

In [ ]:
print("=" * 85)
print("SUMMARY: Our Results vs. Saveski et al. 2025")
print("=" * 85)
print()

pct_pos_views = (cate_results["Views"] > 0).mean() * 100
pct_pos_likes = (cate_results["Likes"] > 0).mean() * 100

rows_table = [
    ("Main effect (Views)",
     "Suppression -46% reposts, -44% likes",
     "Anti-suppression; see main analysis"),
    ("% posts with increased views",
     "43% showed increased views post-note",
     f"{pct_pos_views:.1f}% of Treatment tweets have CATE > 0 (Views)"),
    ("% posts with increased likes",
     "not reported separately",
     f"{pct_pos_likes:.1f}% of Treatment tweets have CATE > 0 (Likes)"),
    ("Growth vs. Overall",
     "Both framings show suppression",
     "See comparison table above"),
    ("Faster intervention",
     "Faster notes suppress MORE",
     "See lag quartile table above"),
]

print(f"  {'Analysis':<38} {'Saveski 2025':<32} {'Our experiment'}")
print("-" * 100)
for analysis, saveski, ours in rows_table:
    print(f"  {analysis:<38} {saveski:<32} {ours}")
    print()

print("Saveski 2025: Community Notes labels — synthetic control — 40k+ posts")
print("Our study:    Human-crafted CN replies via Nakama avatars — RCT — 1,210 tweets")